# Governance & Compliance Audit

This notebook queries the OCP audit SQLite datastore to present findings for:

- **OCP-15 — Editorial Policy, Standards, Procedures**: OpenShift-specific standards, procedures, and controls are published & maintained in an official repository. Evidence on-cluster: governance / policy tooling installed (Gatekeeper, Kyverno, Compliance Operator, ACS / StackRox, ACM, Quay) and OPA Gatekeeper ConstraintTemplates / Constraints with violation counts.
- **OCP-16 — Industry Framework Alignment**: Enterprise standards and controls for OpenShift are mapped to industry standards and best practices (NIST 800-53, OWASP). Evidence on-cluster: Compliance Operator install posture and a static OCP-ID → framework mapping table.
- **OCP-17 — OpenShift Usage Policies**: Enterprise standards must define and document approved and unapproved OpenShift distributions and use cases. Evidence on-cluster: OCP version / channel, infrastructure platform, OperatorHub catalog sources (Red Hat vs community), Subscription install-plan approval mode, and degraded / unavailable cluster operators.
- **OCP 4.18 / 4.19 Version Validation**: Per-cluster check that `ocp_version` is on a supported minor (`4.18` or `4.19`) and `update_channel` is on an approved `stable-*` or `eus-*` train.
- **OCP-20 — Exception Management**: Policies and standards exceptions are tracked, granted timely, and applied consistently. Evidence on-cluster: Gatekeeper Constraints whose `enforcement_action` is not `deny` (i.e. `dryrun` / `warn`) — de facto operational exceptions to a published policy.


In [ ]:
import os
import sys

import pandas as pd

# Ensure the repo root is importable before loading project modules.
sys.path.insert(0, os.path.dirname(os.path.abspath("__file__")))

from notebook_style import bootstrap, style_table  # noqa: E402

print("python:", sys.executable)
print("cwd:", os.getcwd())

# bootstrap() adds ../datastore to sys.path, which is required before
# importing schema.models below.
session, engine = bootstrap()

from schema.models import (  # noqa: E402
    Cluster,
    GovernancePolicyEcosystem,
    OlmGovernance,
    PlatformGuardrail,
    PolicyAsCodeConstraint,
)

print("Connected to:", engine.url)

## Cluster Inventory

In [ ]:
df_clusters = pd.read_sql(
    session.query(
        Cluster.id,
        Cluster.cluster_name,
        Cluster.cluster_context,
        Cluster.cluster_server,
    ).statement,
    engine,
)
print(f"{len(df_clusters)} cluster(s) in dataset")
style_table(df_clusters)

---
## OCP-15: Editorial Policy, Standards, Procedures

*Standards, procedures, and controls are documented and maintained in an official repository.*

On-cluster evidence is the presence and configuration of the tooling that **enforces** those documented standards: a policy engine (OPA Gatekeeper / Kyverno), a compliance scanner (Compliance Operator), and / or a runtime security platform (ACS / StackRox, ACM, Quay).

### Governance & Policy Ecosystem (per cluster)

In [ ]:
df_ecosystem = pd.read_sql(
    session.query(
        Cluster.cluster_name,
        GovernancePolicyEcosystem.record_type,
        GovernancePolicyEcosystem.product_name,
        GovernancePolicyEcosystem.installed,
        GovernancePolicyEcosystem.namespace,
        GovernancePolicyEcosystem.operator_version,
        GovernancePolicyEcosystem.detail_1,
        GovernancePolicyEcosystem.detail_2,
        GovernancePolicyEcosystem.detail_3,
    )
    .join(Cluster, GovernancePolicyEcosystem.cluster_id == Cluster.id)
    .order_by(Cluster.cluster_name, GovernancePolicyEcosystem.product_name)
    .statement,
    engine,
)
style_table(df_ecosystem)

### OPA Gatekeeper Constraints (per cluster)

Evidence that documented policies are enforced as code. A constraint with `enforcement_action = deny` and `total_violations = 0` indicates an actively-enforced rule with no current breaches; `warn` is advisory only.

In [ ]:
df_constraints = pd.read_sql(
    session.query(
        Cluster.cluster_name,
        PolicyAsCodeConstraint.gatekeeper_installed,
        PolicyAsCodeConstraint.constraint_template,
        PolicyAsCodeConstraint.constraint_name,
        PolicyAsCodeConstraint.enforcement_action,
        PolicyAsCodeConstraint.total_violations,
        PolicyAsCodeConstraint.match_kinds,
        PolicyAsCodeConstraint.match_namespaces,
    )
    .join(Cluster, PolicyAsCodeConstraint.cluster_id == Cluster.id)
    .order_by(
        Cluster.cluster_name,
        PolicyAsCodeConstraint.constraint_template,
        PolicyAsCodeConstraint.constraint_name,
    )
    .statement,
    engine,
)
style_table(df_constraints)

### OCP-15 Compliance Flags

- `has_policy_engine` — at least one of Gatekeeper / Kyverno is installed (mandatory for policy-as-code enforcement)
- `has_compliance_scanner` — Compliance Operator is installed (mandatory for periodic CIS / NIST scans)
- `has_active_constraints` — at least one Gatekeeper Constraint exists with `enforcement_action != ''`
- `has_unresolved_violations` — at least one Gatekeeper Constraint reports `total_violations > 0` (informational, not necessarily a fail)

In [ ]:
POLICY_ENGINES = {"gatekeeper", "kyverno"}
COMPLIANCE_PRODUCTS = {"compliance-operator"}

eco_installed = df_ecosystem[df_ecosystem["installed"] == True]  # noqa: E712

rows = []
for cluster_name in df_clusters["cluster_name"]:
    eco = eco_installed[eco_installed["cluster_name"] == cluster_name]
    cons = df_constraints[df_constraints["cluster_name"] == cluster_name]

    has_engine = bool(eco["product_name"].str.lower().isin(POLICY_ENGINES).any())
    has_scanner = bool(eco["product_name"].str.lower().isin(COMPLIANCE_PRODUCTS).any())
    has_active = bool(
        (
            (cons["enforcement_action"].fillna("") != "")
            & cons["gatekeeper_installed"].fillna(False)
        ).any()
    )
    has_violations = bool((cons["total_violations"].fillna(0) > 0).any())

    rows.append(
        {
            "cluster_name": cluster_name,
            "has_policy_engine": has_engine,
            "has_compliance_scanner": has_scanner,
            "has_active_constraints": has_active,
            "has_unresolved_violations": has_violations,
        }
    )

df_ocp15_flags = pd.DataFrame(rows)
style_table(df_ocp15_flags)

---
## OCP-16: Industry Framework Alignment

*Enterprise standards and controls for OpenShift are mapped to industry standards (NIST 800-53, OWASP).*

The mapping itself is policy documentation — it is recorded statically below to mirror the OpenShift Security Scorecard. The on-cluster evidence is the **Compliance Operator** install posture, which provides automated NIST / CIS scanning.

In [ ]:
framework_mapping = [
    (
        "OCP-6",
        "Platform Usage Guardrails",
        "CM-2, CM-6",
        "OWASP K01 Insecure Workload Configurations",
    ),
    (
        "OCP-7",
        "Policy-as-Code Enforcement",
        "AC-3, CM-7",
        "OWASP K02 Supply Chain Vulnerabilities",
    ),
    (
        "OCP-8",
        "CI/CD Pipeline Enforcement",
        "SA-15, CM-7",
        "OWASP K02 Supply Chain Vulnerabilities",
    ),
    (
        "OCP-9",
        "Control Plane Protections",
        "SC-8, SC-12, SC-13",
        "OWASP K05 Inadequate Logging & Monitoring",
    ),
    (
        "OCP-10",
        "Patch & Version Lifecycle Mgmt",
        "SI-2, CM-3",
        "OWASP K07 Outdated & Vulnerable Components",
    ),
    (
        "OCP-11",
        "Secrets & Cert Rotation",
        "SC-12, IA-5",
        "OWASP K06 Broken Authentication & Authorization",
    ),
    (
        "OCP-12",
        "Disaster Recovery & Cluster Backup",
        "CP-9, CP-10",
        "OWASP K10 Inadequate Resilience",
    ),
    ("OCP-13", "OLM Control", "CM-7, SA-12", "OWASP K02 Supply Chain Vulnerabilities"),
    ("OCP-14", "Encryption At Rest", "SC-28", "OWASP K08 Secrets Management Failures"),
    (
        "OCP-15",
        "Editorial Policy, Standards, Procs",
        "SC-28, CM-2, CM-6",
        "OWASP K05 Inadequate Logging & Monitoring",
    ),
    ("OCP-16", "Industry Framework Alignment", "AC-1, CM-2", "OWASP Top 10 mapping"),
    (
        "OCP-17",
        "OpenShift Usage Policies",
        "CM-2, CM-7",
        "OWASP K01 Insecure Workload Configurations",
    ),
    (
        "OCP-20",
        "Exception Management",
        "CA-5, CM-3, PL-4",
        "OWASP K05 Inadequate Logging & Monitoring",
    ),
]

df_frameworks = pd.DataFrame(
    framework_mapping,
    columns=["req_id", "title", "nist_800_53", "owasp"],
)
style_table(df_frameworks)

In [ ]:
rows = []
for cluster_name in df_clusters["cluster_name"]:
    eco = df_ecosystem[
        (df_ecosystem["cluster_name"] == cluster_name)
        & (df_ecosystem["product_name"].str.lower().isin(COMPLIANCE_PRODUCTS))
    ]
    installed_row = eco[eco["installed"] == True]  # noqa: E712
    rows.append(
        {
            "cluster_name": cluster_name,
            "compliance_operator_installed": bool(len(installed_row)),
            "compliance_operator_version": (
                installed_row["operator_version"].iloc[0] if len(installed_row) else ""
            ),
            "compliance_operator_namespace": (
                installed_row["namespace"].iloc[0] if len(installed_row) else ""
            ),
        }
    )

df_ocp16_flags = pd.DataFrame(rows)
style_table(df_ocp16_flags)

---
## OCP-17: OpenShift Usage Policies

*Enterprise standards must define and document approved and unapproved OpenShift distributions and use cases.*

On-cluster evidence:

1. **Distribution** — OCP version, update channel, and infrastructure platform (an unapproved distribution like vanilla Kubernetes, EKS, or AKS would not appear here at all).
2. **Catalog sources** — only Red Hat sources should be present; community catalogs indicate unapproved operator sources.
3. **Subscription approval mode** — production subscriptions should require manual install-plan approval to prevent unapproved upgrades.
4. **Operator health** — degraded or unavailable cluster operators are misuse / misconfiguration signals.

### Distribution & Version (per cluster)

In [ ]:
df_distribution = pd.read_sql(
    session.query(
        Cluster.cluster_name,
        PlatformGuardrail.ocp_version,
        PlatformGuardrail.update_channel,
        PlatformGuardrail.update_state,
        PlatformGuardrail.platform,
        PlatformGuardrail.control_plane_topology,
        PlatformGuardrail.infrastructure_topology,
        PlatformGuardrail.total_operators,
        PlatformGuardrail.degraded_count,
        PlatformGuardrail.unavailable_count,
        PlatformGuardrail.degraded_operators,
        PlatformGuardrail.unavailable_operators,
    )
    .join(Cluster, PlatformGuardrail.cluster_id == Cluster.id)
    .order_by(Cluster.cluster_name)
    .statement,
    engine,
)
style_table(df_distribution)

### Catalog Sources & Subscription Approval Mode

From `olm-governance` exports — `record_type=catalogsource` shows the publisher and image of every operator catalog; `record_type=subscription` shows the install-plan approval mode (`Automatic` vs `Manual`).

In [ ]:
df_catalogs = pd.read_sql(
    session.query(
        Cluster.cluster_name,
        OlmGovernance.name.label("catalog_name"),
        OlmGovernance.namespace,
        OlmGovernance.detail_1.label("publisher"),
        OlmGovernance.detail_2.label("image"),
        OlmGovernance.detail_4.label("state"),
    )
    .join(Cluster, OlmGovernance.cluster_id == Cluster.id)
    .filter(OlmGovernance.record_type == "catalogsource")
    .order_by(Cluster.cluster_name, OlmGovernance.name)
    .statement,
    engine,
)
style_table(df_catalogs)

In [ ]:
df_subs = pd.read_sql(
    session.query(
        Cluster.cluster_name,
        OlmGovernance.name.label("subscription_name"),
        OlmGovernance.namespace,
        OlmGovernance.detail_1.label("channel"),
        OlmGovernance.detail_2.label("source"),
        OlmGovernance.detail_3.label("install_plan_approval"),
    )
    .join(Cluster, OlmGovernance.cluster_id == Cluster.id)
    .filter(OlmGovernance.record_type == "subscription")
    .order_by(Cluster.cluster_name, OlmGovernance.name)
    .statement,
    engine,
)
style_table(df_subs)

### OCP-17 Compliance Flags

- `on_supported_channel` — `update_channel` is on the supported `stable-*` or `eus-*` train for OCP 4.18/4.19
- `redhat_only_catalogs` — every catalog source publisher contains `Red Hat`
- `auto_approval_subscriptions` — count of subscriptions whose install-plan approval is `Automatic` (these should generally be `Manual` in production)
- `no_degraded_operators` — `degraded_count == 0` and `unavailable_count == 0`

In [ ]:
SUPPORTED_OCP_CHANNELS = {"stable", "eus"}
SUPPORTED_OCP_MINORS = {"4.18", "4.19"}


def _channel_supported_for_release(channel: str, ocp_version: str) -> bool:
    """Return True when the channel is valid for OCP 4.18/4.19 policy checks."""
    normalized = (channel or "").strip().lower()
    if not normalized:
        return False

    train = normalized.split("-", 1)[0]
    if train not in SUPPORTED_OCP_CHANNELS:
        return False

    if "-" not in normalized:
        return False

    channel_minor = normalized.split("-", 1)[1].strip()
    if not channel_minor:
        return False

    # Prefer explicit version data when available, otherwise fall back to the channel suffix.
    v = str(ocp_version or "").strip()
    version_minor = ".".join(v.split(".")[:2]) if v else ""
    effective_minor = version_minor or channel_minor
    return effective_minor in SUPPORTED_OCP_MINORS


rows = []
for _, dist in df_distribution.iterrows():
    cluster_name = dist["cluster_name"]

    channel = str(dist.get("update_channel") or "")
    ocp_version = str(dist.get("ocp_version") or "")
    on_supported_channel = _channel_supported_for_release(channel, ocp_version)

    cats = df_catalogs[df_catalogs["cluster_name"] == cluster_name]
    if len(cats):
        redhat_only = bool(
            cats["publisher"].fillna("").str.contains("Red Hat", case=False).all()
        )
    else:
        redhat_only = False

    sub_set = df_subs[df_subs["cluster_name"] == cluster_name]
    auto_count = int(
        (
            sub_set["install_plan_approval"]
            .fillna("")
            .str.contains("Automatic", case=False)
        ).sum()
    )

    no_degraded = (int(dist["degraded_count"] or 0) == 0) and (
        int(dist["unavailable_count"] or 0) == 0
    )

    rows.append(
        {
            "cluster_name": cluster_name,
            "ocp_version": dist["ocp_version"],
            "update_channel": dist["update_channel"],
            "on_supported_channel": on_supported_channel,
            "redhat_only_catalogs": redhat_only,
            "auto_approval_subscriptions": auto_count,
            "no_degraded_operators": no_degraded,
        }
    )

df_ocp17_flags = pd.DataFrame(rows)
style_table(df_ocp17_flags)

---
## OCP 4.18 / 4.19 Version Validation

*Executes the explicit version-and-channel check declared at the top of this notebook.*

This section evaluates each cluster's `ocp_version` and `update_channel` against the
approved distributions (**OCP 4.18** and **OCP 4.19** on `stable-*` or `eus-*` channels)
and surfaces any cluster that is **not** on a supported release.

The same `_channel_supported_for_release` helper used by the OCP-17 flags cell is
re-applied here so the result is auditable in isolation.

In [ ]:
SUPPORTED_OCP_CHANNELS = {"stable", "eus"}
SUPPORTED_OCP_MINORS = {"4.18", "4.19"}


def _channel_supported_for_release(channel: str, ocp_version: str) -> bool:
    """Return True when the channel is valid for OCP 4.18/4.19 policy checks."""
    normalized = (channel or "").strip().lower()
    if not normalized:
        return False

    train = normalized.split("-", 1)[0]
    if train not in SUPPORTED_OCP_CHANNELS:
        return False

    if "-" not in normalized:
        return False

    channel_minor = normalized.split("-", 1)[1].strip()
    if not channel_minor:
        return False

    v = str(ocp_version or "").strip()
    version_minor = ".".join(v.split(".")[:2]) if v else ""
    effective_minor = version_minor or channel_minor
    return effective_minor in SUPPORTED_OCP_MINORS


version_rows = []
for _, dist in df_distribution.iterrows():
    cluster_name = dist["cluster_name"]
    channel = str(dist.get("update_channel") or "")
    ocp_version = str(dist.get("ocp_version") or "")

    v = ocp_version.strip()
    minor = ".".join(v.split(".")[:2]) if v else ""
    on_supported_minor = minor in SUPPORTED_OCP_MINORS
    on_supported_channel = _channel_supported_for_release(channel, ocp_version)

    version_rows.append(
        {
            "cluster_name": cluster_name,
            "ocp_version": ocp_version,
            "ocp_minor": minor,
            "update_channel": channel,
            "on_supported_minor": on_supported_minor,
            "on_supported_channel": on_supported_channel,
            "compliant": on_supported_minor and on_supported_channel,
        }
    )

df_ocp_version_validation = pd.DataFrame(version_rows)

total = len(df_ocp_version_validation)
compliant = int(df_ocp_version_validation["compliant"].sum()) if total else 0
print(
    f"{compliant}/{total} cluster(s) on a supported OCP 4.18/4.19 release "
    f"and stable-*/eus-* channel"
)

style_table(df_ocp_version_validation)

---
## OCP-20: Exception Management

*Policies and standards exceptions are tracked, granted timely, and applied consistently.*

OCP-20 is primarily a **documentation & process control** — exceptions to published
policies should live in a tracked register (e.g. an exception ticket / GRC system).

The closest on-cluster proxy is **OPA Gatekeeper Constraint enforcement mode**:

- A Constraint with `enforcement_action == "deny"` enforces the policy.
- A Constraint with `enforcement_action` of `dryrun` or `warn` is **not blocking
  violations** — it is operating as a *de facto exception* to the published policy.

The cells below summarise, per cluster:

1. Counts of Constraints by enforcement mode.
2. The full list of Constraints that are **not** in `deny` mode (the exception
  register that should be reconciled against the documented policy exceptions).


In [ ]:
EXCEPTION_MODES = {"dryrun", "warn"}


def _normalize_action(value: object) -> str:
    return str(value or "").strip().lower()


# Only consider real Constraint rows — skip the gatekeeper_installed=False placeholder rows
df_real_constraints = df_constraints[
    df_constraints["constraint_name"].fillna("").str.len() > 0
].copy()

if df_real_constraints.empty:
    print(
        "No Gatekeeper Constraints loaded — OCP-20 cannot be evaluated from cluster data."
    )
    df_ocp20_summary = pd.DataFrame()
    df_ocp20_exceptions = pd.DataFrame()
else:
    df_real_constraints["enforcement_mode"] = (
        df_real_constraints["enforcement_action"]
        .map(_normalize_action)
        .replace("", "unset")
    )
    df_real_constraints["is_exception"] = df_real_constraints["enforcement_mode"].isin(
        EXCEPTION_MODES
    )

    summary_rows = []
    for cluster_name, grp in df_real_constraints.groupby("cluster_name"):
        total = len(grp)
        deny = int((grp["enforcement_mode"] == "deny").sum())
        dryrun = int((grp["enforcement_mode"] == "dryrun").sum())
        warn = int((grp["enforcement_mode"] == "warn").sum())
        other = total - deny - dryrun - warn
        exceptions = dryrun + warn
        summary_rows.append(
            {
                "cluster_name": cluster_name,
                "total_constraints": total,
                "deny": deny,
                "dryrun": dryrun,
                "warn": warn,
                "other_or_unset": other,
                "active_exceptions": exceptions,
                "exception_rate_pct": (
                    round((exceptions / total) * 100, 1) if total else 0.0
                ),
            }
        )
    df_ocp20_summary = pd.DataFrame(summary_rows)

    df_ocp20_exceptions = df_real_constraints[df_real_constraints["is_exception"]][
        [
            "cluster_name",
            "constraint_template",
            "constraint_name",
            "enforcement_mode",
            "total_violations",
        ]
    ].sort_values(["cluster_name", "constraint_template", "constraint_name"])

    print(
        f"{len(df_ocp20_exceptions)} de facto exception(s) "
        f"(Constraints not in deny mode) across "
        f"{df_ocp20_summary['cluster_name'].nunique()} cluster(s)"
    )

print("\nOCP-20 Summary — Constraint enforcement modes per cluster:")
style_table(df_ocp20_summary)

In [ ]:
print("OCP-20 Exception Register — Constraints not in 'deny' mode:")
if df_ocp20_exceptions.empty:
    print("  (none — all Constraints are enforcing 'deny')")
style_table(df_ocp20_exceptions)